In [ ]:
!pip install transformers torch fastapi uvicorn nest-asyncio pyngrok


In [ ]:
from transformers import pipeline
from fastapi import FastAPI
from pydantic import BaseModel
import nest_asyncio
from pyngrok import ngrok
import uvicorn


In [ ]:
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment",
    top_k=None
)

LABEL_MAP = {
    "LABEL_0": "Negative 😠",
    "LABEL_1": "Neutral 😐",
    "LABEL_2": "Positive 😊"
}



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vocab.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

In [ ]:
text = "This project is really impressive"
result = sentiment_pipeline(text)[0]

for item in result:
    print(LABEL_MAP[item["label"]], ":", round(item["score"] * 100, 2), "%")



Positive 😊 : 98.61 %
Neutral 😐 : 1.24 %
Negative 😠 : 0.15 %


In [ ]:
def predict_sentiment(text):
    results = sentiment_pipeline(text)[0]

    # pick label with highest confidence
    best = max(results, key=lambda x: x["score"])

    sentiment = LABEL_MAP[best["label"]]
    confidence = round(best["score"] * 100, 2)

    return sentiment, confidence


In [ ]:
text = "This project is really impressive"
sentiment, confidence = predict_sentiment(text)

print("Text:", text)
print("Sentiment:", sentiment)
print("Confidence:", confidence, "%")


Text: This project is really impressive
Sentiment: Positive 😊
Confidence: 98.61 %


In [ ]:
texts = [
    "This project is really impressive",
    "The app is okay but needs improvement",
    "I hate how slow this system is"
]

for t in texts:
    sentiment, confidence = predict_sentiment(t)
    print(f"Text: {t}")
    print(f"Sentiment: {sentiment} | Confidence: {confidence}%")
    print("-" * 60)


Text: This project is really impressive
Sentiment: Positive 😊 | Confidence: 98.61%
------------------------------------------------------------
Text: The app is okay but needs improvement
Sentiment: Positive 😊 | Confidence: 58.08%
------------------------------------------------------------
Text: I hate how slow this system is
Sentiment: Negative 😠 | Confidence: 98.08%
------------------------------------------------------------


In [ ]:
def predict_with_threshold(text, threshold=60):
    results = sentiment_pipeline(text)[0]
    best = max(results, key=lambda x: x["score"])

    confidence = best["score"] * 100

    if confidence < threshold:
        return "Neutral 😐", round(confidence, 2)

    return LABEL_MAP[best["label"]], round(confidence, 2)


In [ ]:
text = "The movie was fine, nothing special"
sentiment, confidence = predict_with_threshold(text)

print("Sentiment:", sentiment)
print("Confidence:", confidence, "%")


Sentiment: Neutral 😐
Confidence: 44.57 %


In [ ]:
!pip install gradio


In [ ]:
import gradio as gr

def analyze_text(text):
    results = sentiment_pipeline(text)[0]
    best = max(results, key=lambda x: x["score"])

    sentiment = LABEL_MAP[best["label"]]
    confidence = round(best["score"] * 100, 2)

    return f"{sentiment} ({confidence}%)"


In [ ]:
interface = gr.Interface(
    fn=analyze_text,
    inputs=gr.Textbox(lines=3, placeholder="Enter text here..."),
    outputs="text",
    title="BERT Sentiment Analysis System",
    description="Transformer-based sentiment detection using Hugging Face"
)

interface.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2807519ee1d639cee7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:


app = FastAPI(title="BERT Sentiment Analysis API")

class TextInput(BaseModel):
    text: str

@app.post("/predict")
def predict_api(input: TextInput):
    results = sentiment_pipeline(input.text)[0]
    best = max(results, key=lambda x: x["score"])

    return {
        "text": input.text,
        "sentiment": LABEL_MAP[best["label"]],
        "confidence": round(best["score"] * 100, 2)
    }


In [ ]:
# ---------------- MODEL EVALUATION ---------------- #

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Small labeled evaluation dataset (demo purpose)
eval_data = [
    ("I love this product", "Positive 😊"),
    ("This is the worst experience ever", "Negative 😠"),
    ("It is okay, nothing special", "Neutral 😐"),
    ("Absolutely fantastic service", "Positive 😊"),
    ("I hate how slow this app is", "Negative 😠")
]

y_true = []
y_pred = []

for text, true_label in eval_data:
    pred_label, _ = predict_sentiment(text)
    y_true.append(true_label)
    y_pred.append(pred_label)

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average="weighted")
recall = recall_score(y_true, y_pred, average="weighted")
f1 = f1_score(y_true, y_pred, average="weighted")

print("Evaluation Metrics:")
print("Accuracy :", round(accuracy, 2))
print("Precision:", round(precision, 2))
print("Recall   :", round(recall, 2))
print("F1 Score :", round(f1, 2))


Evaluation Metrics:
Accuracy : 0.8
Precision: 0.67
Recall   : 0.8
F1 Score : 0.72


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
